# Phase 21: Exit Model (Model 2) - HIGH-RIGOR TRAINING
=======================================================
This is the **Official Thesis Version** of the training script.

### Key Improvements:
1. **Leakage-Proof**: No transaction events in input.
2. **Full Dataset**: Trains on **1.7 million sessions**.
3. **Scientific Monitoring**: Benchmarks **PR-AUC** (Average Precision).
4. **Early Stopping**: Automated convergence detection.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import sys
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

# Add scripts dir for architecture imports
sys.path.append(os.path.abspath("../scripts"))
from train_tcn import AbandonmentTCN
from train_transformer import ClickstreamDataset

DATA_DIR = "../data/processed"
MODEL_SAVE_PATH = "../backend/models/exit_model_tcn.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 20
BATCH_SIZE = 4096
LR = 1e-3

print(f"🚀 STARTING HIGH-RIGOR TRAINING")
print(f"🚀 DEVICE: {DEVICE}")

🚀 STARTING HIGH-RIGOR TRAINING
🚀 DEVICE: cuda


In [2]:
print("Loading 1.7M session dataset...")
X_page = np.load(os.path.join(DATA_DIR, "X_page_real.npy"))
X_dur = np.load(os.path.join(DATA_DIR, "X_dur_real.npy"))
y = np.load(os.path.join(DATA_DIR, "y_abandon_real.npy"))

print("Creating stratified splits...")
train_idx, temp_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y[temp_idx], random_state=42)

train_loader = DataLoader(ClickstreamDataset(X_page[train_idx], X_dur[train_idx], y[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ClickstreamDataset(X_page[val_idx], X_dur[val_idx], y[val_idx]), batch_size=BATCH_SIZE)

print(f"📊 Total: {len(y):,} | Train: {len(train_idx):,} | Val: {len(val_idx):,}")
print(f"Target Abandonment: {y.mean()*100:.2f}%")

Loading 1.7M session dataset...
Creating stratified splits...
📊 Total: 1,760,979 | Train: 1,408,783 | Val: 176,098
Target Abandonment: 99.23%


In [3]:
model = AbandonmentTCN(num_page_types=4).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val_pr = 0
patience = 3
counter = 0

In [4]:
print("Starting Intensive Training... (Expected runtime: 5-15 mins depending on GPU)")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for page, dur, label in train_loader:
        page, dur, label = page.to(DEVICE), dur.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad()
        logits = model(page, dur).squeeze()
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
            
    model.eval()
    val_probs, val_labels = [], []
    with torch.no_grad():
        for page, dur, label in val_loader:
            page, dur, label = page.to(DEVICE), dur.to(DEVICE), label.to(DEVICE)
            logits = model(page, dur).squeeze()
            probs = torch.sigmoid(logits)
            val_probs.extend(probs.cpu().numpy())
            val_labels.extend(label.cpu().numpy())
    
    val_pr = average_precision_score(val_labels, val_probs)
    print(f"Epoch {epoch+1:02d} | Loss: {train_loss/len(train_loader):.4f} | Val PR-AUC: {val_pr:.4f}")
    
    if val_pr > best_val_pr:
        best_val_pr = val_pr
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"✨ Best Model Updated")
        counter = 0
    else:
        counter += 1
        if counter >= patience: break

print(f"\n✅ FINAL BEST PR-AUC: {best_val_pr:.4f}")

Starting Intensive Training... (Expected runtime: 5-15 mins depending on GPU)
Epoch 01 | Loss: 0.0467 | Val PR-AUC: 0.9994
✨ Best Model Updated
Epoch 02 | Loss: 0.0228 | Val PR-AUC: 0.9994
✨ Best Model Updated
Epoch 03 | Loss: 0.0224 | Val PR-AUC: 0.9994
✨ Best Model Updated
Epoch 04 | Loss: 0.0220 | Val PR-AUC: 0.9994
✨ Best Model Updated
Epoch 05 | Loss: 0.0220 | Val PR-AUC: 0.9994
Epoch 06 | Loss: 0.0218 | Val PR-AUC: 0.9994
Epoch 07 | Loss: 0.0218 | Val PR-AUC: 0.9994
✨ Best Model Updated
Epoch 08 | Loss: 0.0218 | Val PR-AUC: 0.9994
Epoch 09 | Loss: 0.0218 | Val PR-AUC: 0.9994
Epoch 10 | Loss: 0.0216 | Val PR-AUC: 0.9994

✅ FINAL BEST PR-AUC: 0.9994
